# Introduction

Monte Carlo Tree Search combines two ideas:
- **Evaluation by Rollouts:** Play multiple games to termination from a state s (using a simple, fast rollout policy) and count wins and losses
- **Selective search:** Explore parts of the tree that will help improve the decision at the root, regardless of depth

For each **rollout**:
- Repeat until terminal:
    - Play a move according to a fixed, fast rollout policy
- Record the result

Fraction of wins correlates with the true value of the position

Having a better rollout policy helps

# Core Mechanism of MCTS

Repeat until out of time:
- Given the current search tree, recursively apply UCB to choose a path down to a leaf (not fully expanded) node $n$
- Add a new child $c$ to $n$ and run a rollout from $c$
- Update the win counts from $c$ back up to the root

Choose the action leading to the child with highest $N$

MCTS runs as many simulations (rollouts) as possible within the allowed time or computational budget.  Each simulation consists of four phases: **Selection**, **Expansion**, **Simulation**, and **Backpropagation**.

## Selection:

**UCB Heuristics**

UCB1 formula combines **promising** and **uncertain**:
$$UCB1(n) = \underbrace{\frac{U(n)}{N(n)}}_{\text{exploitation}} + \underbrace{c \sqrt{\frac{\ln N(Parent(n))}{N(n)}}}_{\text{exploration}}$$

Where:
- $U(n)$: Total utility of node $n$
- $N(n)$: Total Visit count of node $n$
- $N(Parent(n))$: Total visit count of the parent of node $n$
- $c$: Exploration constant *(typically $\sqrt{2}$ or adjusted based on the problem)*


*Select the child node with the highest UCT value, prioritizing nodes with high average rewards (exploitation) and low visit counts (exploration).*

## Expansion

Once a leaf node is found that’s not terminal and not fully expanded, we expand it by adding a new child $c$ to represent an unvisited action.

**Leaf Node:**  
In MCTS, a leaf node is a node where the **search has stopped** during Selection.

It can be either:
- **Terminal:** no further actions possible (e.g., game over).
- **Non-terminal but not fully expanded:** there are still possible actions from this state that haven’t been added as children yet.

**Not fully expanded** means:  
Not all possible actions from node $n$ have been tried (i.e., not all children have been created).

So, in the **Expansion Step**:
- You choose one of the untried actions
- You create a new child node $c$ that represents the state resulting from untried action
- This new node $c$ is added as a child of $n$

## Simulation

- From the newly added child $c$, simulate a complete game (random or heuristic-based rollout) to get an estimated outcome (e.g., win/loss, score).

- This is a **Monte Carlo simulation**.



## Backpropagation

Update the win counts from $c$ **back up to the root**

Take the result of the rollout and **backpropagate** it through the tree:

For each node on the path from $c$ to the root, update:
- Visit count: $N(n)$
- Value sum: $U(n)$

# Codes

## Imports

In [3]:
import math
import random
from abc import ABC, abstractmethod
from typing import List, Optional, Any

## GameState Abstract Class

This defines the required interface for any game to be used with MCTS.

In [4]:
class GameState(ABC):
    """Abstract base class for game states"""
    
    @abstractmethod
    def get_legal_actions(self) -> List[Any]:
        """Return list of legal actions from this state"""
        pass
    
    @abstractmethod
    def apply_action(self, action: Any) -> 'GameState':
        """Return new state after applying action"""
        pass
    
    @abstractmethod
    def is_terminal(self) -> bool:
        """Return True if this is a terminal state"""
        pass
    
    @abstractmethod
    def get_reward(self, player: int) -> float:
        """Return reward for given player (1 or -1)"""
        pass
    
    @abstractmethod
    def get_current_player(self) -> int:
        """Return current player (1 or -1)"""
        pass

## MCTSNode Class

Each node tracks game state, statistics, children, and untried actions.

Recall:
**UCB Heuristics:**
$$UCB1(n) = \underbrace{\frac{U(n)}{N(n)}}_{\text{exploitation}} + \underbrace{c \sqrt{\frac{\ln N(Parent(n))}{N(n)}}}_{\text{exploration}}$$

In [ ]:
class MCTSNode:
    """Node in the MCTS tree"""
    
    def __init__(self, state: GameState, parent: Optional['MCTSNode'] = None, action: Any = None):
        self.state = state
        self.parent = parent
        self.action = action
        self.children: List['MCTSNode'] = []
        self.visits = 0
        self.wins = 0.0
        self.untried_actions = state.get_legal_actions().copy()
        self.player = state.get_current_player()
    
    def is_fully_expanded(self) -> bool:
        """Check if all actions have been tried"""
        return len(self.untried_actions) == 0
    
    def is_terminal(self) -> bool:
        """Check if this is a terminal node"""
        return self.state.is_terminal()
    
    def ucb1_value(self, exploration_param: float = math.sqrt(2)) -> float:
        """Calculate UCB1 value for this node"""
        if self.visits == 0:
            return float('inf')
        
        exploitation = self.wins / self.visits
        exploration = exploration_param * math.sqrt(math.log(self.parent.visits) / self.visits)
        return exploitation + exploration
    
    def select_child(self, exploration_param: float = math.sqrt(2)) -> 'MCTSNode':
        """Select child with highest UCB1 value"""
        return max(self.children, key=lambda child: child.ucb1_value(exploration_param))
    
    def expand(self) -> 'MCTSNode':
        """Expand tree by adding a new child node"""
        action = self.untried_actions.pop()
        new_state = self.state.apply_action(action)
        child = MCTSNode(new_state, parent=self, action=action)
        self.children.append(child)
        return child
    
    def update(self, result: float):
        """Update node statistics with simulation result"""
        self.visits += 1
        self.wins += result

## UCT Search Algorithm

Upper Confidence Bounds applied to Trees (UCT) - the core Monte Carlo Tree Search class.

It manages search iterations and contains logic for **selection**, **expansion**, **simulation**, and **backpropagation**.

In [6]:
class UCT:
    """UCT (Upper Confidence bounds applied to Trees) algorithm for MCTS"""
    
    def __init__(self, exploration_param: float = math.sqrt(2)):
        self.exploration_param = exploration_param
    
    def search(self, root_state: GameState, iterations: int = 1000) -> Any:
        """
        Perform UCT search and return the best action
        
        Args:
            root_state: Initial game state
            iterations: Number of MCTS iterations to perform
            
        Returns:
            Best action to take from root state
        """
        root = MCTSNode(root_state)
        
        for _ in range(iterations):
            # Selection and Expansion
            node = self._select_and_expand(root)
            
            # Simulation
            result = self._simulate(node.state)
            
            # Backpropagation
            self._backpropagate(node, result)
        
        # Return action of most visited child
        if not root.children:
            return random.choice(root_state.get_legal_actions())
        
        best_child = max(root.children, key=lambda child: child.visits)
        return best_child.action
    
    def _select_and_expand(self, root: MCTSNode) -> MCTSNode:
        """Selection and expansion phases of MCTS"""
        node = root
        
        # Selection: traverse down the tree using UCB1
        while not node.is_terminal() and node.is_fully_expanded():
            node = node.select_child(self.exploration_param)
        
        # Expansion: add a new child if possible
        if not node.is_terminal() and not node.is_fully_expanded():
            node = node.expand()
        
        return node
    
    def _simulate(self, state: GameState) -> float:
        """
        Simulation phase - random playout from given state
        
        Returns:
            Reward for the player who made the root move
        """
        current_state = state
        
        while not current_state.is_terminal():
            actions = current_state.get_legal_actions()
            action = random.choice(actions)
            current_state = current_state.apply_action(action)
        
        # Return reward from perspective of root player
        return current_state.get_reward(1)  # Assuming player 1 perspective
    
    def _backpropagate(self, node: MCTSNode, result: float):
        """Backpropagation phase - update all ancestors"""
        while node is not None:
            # Flip result for alternating players
            node_result = result if node.player == 1 else -result
            node.update(node_result)
            node = node.parent

## Tic-Tac-Toe Game Implementation

In [7]:
# Example usage with a simple Tic-Tac-Toe game state
class TicTacToeState(GameState):
    """Simple Tic-Tac-Toe implementation for demonstration"""
    
    def __init__(self, board: Optional[List[List[int]]] = None, current_player: int = 1):
        self.board = board if board else [[0 for _ in range(3)] for _ in range(3)]
        self.current_player = current_player
    
    def get_legal_actions(self) -> List[tuple]:
        """Return list of (row, col) positions that are empty"""
        actions = []
        for i in range(3):
            for j in range(3):
                if self.board[i][j] == 0:
                    actions.append((i, j))
        return actions
    
    def apply_action(self, action: tuple) -> 'TicTacToeState':
        """Apply move and return new state"""
        row, col = action
        new_board = [row[:] for row in self.board]
        new_board[row][col] = self.current_player
        return TicTacToeState(new_board, -self.current_player)
    
    def is_terminal(self) -> bool:
        """Check if game is over"""
        return self._check_winner() != 0 or len(self.get_legal_actions()) == 0
    
    def get_reward(self, player: int) -> float:
        """Get reward for specified player"""
        winner = self._check_winner()
        if winner == player:
            return 1.0
        elif winner == -player:
            return -1.0
        else:
            return 0.0
    
    def get_current_player(self) -> int:
        return self.current_player
    
    def _check_winner(self) -> int:
        """Check for winner, return 1, -1, or 0"""
        # Check rows, columns, and diagonals
        for i in range(3):
            if abs(sum(self.board[i])) == 3:
                return self.board[i][0]
            if abs(sum(self.board[j][i] for j in range(3))) == 3:
                return self.board[0][i]
        
        # Check diagonals
        if abs(sum(self.board[i][i] for i in range(3))) == 3:
            return self.board[0][0]
        if abs(sum(self.board[i][2-i] for i in range(3))) == 3:
            return self.board[0][2]
        
        return 0


In [10]:
# Example usage
if __name__ == "__main__":
    # Create initial game state
    initial_state = TicTacToeState()
    
    # Create UCT agent
    uct = UCT(exploration_param=1.4)
    
    # Find best move
    best_action = uct.search(initial_state, iterations=1000)
    print(f"Best action: {best_action}")
    
    # You can also access the search tree for analysis
    print("UCT search completed!")

Best action: (0, 0)
UCT search completed!


# References

Berkeley University AI lecture slides